# WhisperColab

Google Colab 무료 GPU에서 **Whisper Large-v3**로 음성 파일을 TXT로 변환합니다. **Colab 기본 업로드 창**을 사용해 파일을 Python에 전달하므로, 업로드 개수만 표시되고 멈추는 \`ipywidgets.FileUpload\` 문제를 피합니다.

1. **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. **STEP 1** 설치 셀 실행
3. **STEP 2**에서 언어와 무음 감지를 고르고 **▶ 실행** → 파일 업로드
4. 업로드가 끝나면 자동으로 변환하고 **TXT를 자동 다운로드**합니다.
5. 다른 음성도 변환하려면 **STEP 2 셀만 다시 실행**하세요.

**외부 웹서버나 TryCloudflare를 사용하지 않습니다.** GPU 할당 및 런타임 지속 시간은 Colab 정책에 따라 제한됩니다.

> 예전 UI의 '음성 파일 선택 (1)'에서 진행되지 않는 경우, 오래된 노트북 탭을 닫고 이 노트북을 새로 열어 진행해 주세요.


**변환 오류 수정 (2026-10-30):** `metadata_errors` 오류는 PyAV 18/19의 API 변경 때문입니다. STEP 1을 다시 실행하면 호환 버전 `av==17.1.0`을 설치하고 실제 오디오 파일 디코딩까지 검증합니다. HF_TOKEN 메시지는 다운로드 속도/한도 경고로 변환 실패 원인이 아닙니다.


**M4A 빈 TXT 수정:** 음성 파일을 16kHz PCM으로 검사·변환한 후 인식하고, 결과가 없으면 VAD 해제 재시도합니다. 음성이 끝까지 인식되지 않으면 TXT를 내려받지 않고 이유를 표시합니다. 기본 VAD는 꺼져 있습니다.

**강의 인식 품질 개선:** 문맥 누적 반복을 줄이기 위해 이전 문맥 이어받기를 끄고, 보수적인 VAD를 기본 켭니다. GPU 정확도 우선 모드(`float16`) 및 전문용어 힌트를 추가했습니다. 실제 말한 반복은 무조건 삭제하지 않습니다.



In [ ]:
# @title STEP 1 · 무료 GPU 확인 및 설치 (최초 한 번만 ▶)
import subprocess
import sys
from pathlib import Path

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True, capture_output=True,
)
if gpu.returncode:
    raise RuntimeError("GPU가 없습니다. '런타임 → 런타임 유형 변경 → T4 GPU'를 선택하세요.")
print("✅ 연결된 GPU:", gpu.stdout.strip())

repo_dir = Path("/content/WhisperColab")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/dariring/WhisperColab.git", str(repo_dir)],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "requirements.txt")],
    check=True,
)
# 새 Python 프로세스에서 실사용 오디오 디코더와 PyAV 호환성을 검사합니다.
# (같은 Colab 세션에서 이미 PyAV 19가 import돼 있어도 올바르게 검증됩니다.)
decoder_check = """
from io import BytesIO
import wave
import av
import importlib.metadata

audio = BytesIO()
with wave.open(audio, "wb") as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(16000)
    wav.writeframes(b"\\x00\\x00" * 1600)

# faster-whisper 1.2.1이 실제로 호출하는 API 형태
with av.open(BytesIO(audio.getvalue()), mode="r", metadata_errors="ignore") as source:
    assert source.streams.audio, "오디오 트랙을 읽을 수 없습니다"
print("✅ 오디오 디코더 호환성 확인:", "PyAV", av.__version__,
      "/ faster-whisper", importlib.metadata.version("faster-whisper"))
"""
subprocess.run([sys.executable, "-c", decoder_check], check=True)
# M4A 디코딩은 Colab에 포함된 ffmpeg/ffprobe를 사용합니다.
from shutil import which
if which("ffmpeg") is None or which("ffprobe") is None:
    raise RuntimeError("ffmpeg 또는 ffprobe를 찾지 못했습니다. Colab 기본 런타임에서 다시 시도해 주세요.")
print("✅ M4A/AAC 오디오 디코더(ffmpeg) 준비 완료")
print("✅ 설치 완료. STEP 2의 ▶ 실행 버튼을 누르고 파일을 업로드하세요.")





In [ ]:
# @title STEP 2 · 파일 업로드 → 자동 변환 → TXT 다운로드 (변환할 때마다 ▶)
# @markdown Colab 노트북의 **실행 버튼(▶)** 을 누르면 파일 선택창이 뜹니다.
언어 = "한국어" # @param ["자동 감지", "한국어", "English", "日本語", "中文"]
무음_구간_제외 = True # @param {type:"boolean"}
# @markdown 강의 고유명사를 입력하면 인식에 힌트를 줄 수 있습니다. 다른 음성이라면 비워 주세요.
전문용어_힌트 = "공자, 안회, 자로, 자공, 논어, 맹자" # @param {type:"string"}
GPU_정밀도 = "float16" # @param ["float16", "int8_float16"]
파일_가져오기 = "업로드 창 열기" # @param ["업로드 창 열기", "Colab 파일 경로 사용"]
Colab_파일_경로 = "" # @param {type:"string"}
자동_TXT_다운로드 = True # @param {type:"boolean"}

import html
import json
import os
import re
import subprocess
import sys
import uuid
from pathlib import Path

import nvidia.cublas
import nvidia.cudnn
from google.colab import files
from IPython.display import HTML, display

# Google Colab 내장 파일 업로드 및 다운로드: ipywidgets의 비동기 이벤트에 의존하지 않습니다.
language_codes = {
    "자동 감지": "auto",
    "한국어": "ko",
    "English": "en",
    "日本語": "ja",
    "中文": "zh",
}
allowed_extensions = {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".opus",
                      ".aac", ".mp4", ".webm", ".wma", ".aiff", ".mkv"}
max_bytes = 75 * 1024 * 1024
result_dir = Path("/content/whispercolab/results")
result_dir.mkdir(parents=True, exist_ok=True)

try:
    repo_dir
except NameError:
    raise RuntimeError("먼저 STEP 1 셀을 실행해 주세요.")

from_upload = 파일_가져오기 == "업로드 창 열기"
if from_upload:
    print("📁 음성 파일을 선택해 주세요. (선택하면 자동으로 변환합니다.)")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("선택한 파일이 없습니다. 다시 실행해 파일을 선택해 주세요.")
    if len(uploaded) != 1:
        raise RuntimeError("파일은 한 번에 하나만 올려 주세요.")
    selected_name = next(iter(uploaded))
    input_path = Path(selected_name).resolve()
    # Colab native upload saves the file as well as returning its bytes.
    # No need to hold a second copy in memory.
    del uploaded
else:
    input_path = Path(Colab_파일_경로.strip()).expanduser().resolve()
    if not Colab_파일_경로.strip() or not input_path.is_file():
        raise FileNotFoundError("Colab 파일 경로가 올바르지 않습니다. 왼쪽 파일 탭에서 경로를 복사하세요.")

if input_path.suffix.lower() not in allowed_extensions:
    raise ValueError("지원하지 않는 파일 형식입니다. MP3, WAV, M4A, FLAC, MP4 등을 선택하세요.")
file_size = input_path.stat().st_size
if file_size <= 0 or file_size > max_bytes:
    raise ValueError("0바이트이거나 75MB를 초과하는 파일입니다.")

clean_name = re.sub(r"[^\w .()-]", "_", input_path.stem).strip(" ._")[:75] or "transcript"
txt_path = result_dir / (clean_name + "-" + uuid.uuid4().hex[:8] + ".txt")
libs = [
    str(Path(nvidia.cublas.__path__[0]) / "lib"),
    str(Path(nvidia.cudnn.__path__[0]) / "lib"),
]
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = os.pathsep.join(
    libs + ([env["LD_LIBRARY_PATH"]] if env.get("LD_LIBRARY_PATH") else [])
)
env["HF_HUB_DISABLE_TELEMETRY"] = "1"

print(f"🎙️ 파일: {input_path.name} · {file_size / (1024 * 1024):.1f}MB · 언어: {언어}")
print(f"⚙️ GPU: {GPU_정밀도} · VAD: {'ON' if 무음_구간_제외 else 'OFF'} · 문맥 이어받기: OFF")
if 전문용어_힌트.strip():
    print("📚 전문용어 힌트:", 전문용어_힌트.strip())
display_handle = display(
    HTML('<div style="padding:14px;border:1px solid #cdd5e1;border-radius:9px">'
         '<b>Whisper Large-v3</b><p>변환 준비 중…</p></div>'),
    display_id=True,
)
def show_progress(message, percentage=None):
    fraction = "" if percentage is None else f"<div style='width:100%;background:#e5eaf1;border-radius:5px;height:7px'><div style='background:#5587ce;border-radius:5px;width:{max(0, min(100, int(percentage)))}%;height:7px'></div></div>"
    display_handle.update(HTML(
        "<div style='padding:14px;border:1px solid #cdd5e1;border-radius:9px'>"
        "<b>Whisper Large-v3</b>"
        f"<p>{html.escape(str(message))}</p>{fraction}</div>"
    ))

args = [
    sys.executable, "-u", str(repo_dir / "whisper_worker.py"),
    "--input", str(input_path),
    "--output", str(txt_path),
    "--language", language_codes[언어],
    "--hotwords", 전문용어_힌트.strip(),
    "--compute-type", GPU_정밀도,
]
if not 무음_구간_제외:
    args.append("--no-vad")

detected_language = "?"
last_errors = []
return_code = None
try:
    # Executed synchronously in this notebook cell: output and errors are always visible.
    with subprocess.Popen(
        args, cwd=str(repo_dir), env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    ) as proc:
        assert proc.stdout is not None
        for line in proc.stdout:
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                last_errors.append(line.rstrip())
                last_errors = last_errors[-12:]
                continue
            kind = event.get("kind")
            if kind == "status":
                show_progress(event.get("message", "작업 중…"))
            elif kind == "progress":
                show_progress(event.get("message", "인식 중…"), event.get("percent", 0))
            elif kind == "audio":
                print("🔎", event.get("message", "오디오 검사 완료"))
            elif kind == "warning":
                print("⚠️", event.get("message", "오디오 확인 필요"))
                show_progress(event.get("message", "다시 시도 중"))
            elif kind == "done":
                detected_language = event.get("language") or "?"
            elif kind == "error":
                last_errors.append(event.get("message", "알 수 없는 오류"))
        return_code = proc.wait()
finally:
    # Only remove the file created by the native upload; never delete a user-specified path.
    if from_upload:
        input_path.unlink(missing_ok=True)

if return_code != 0 or not txt_path.is_file():
    show_progress("❌ 변환 중 오류가 발생했습니다")
    raise RuntimeError("변환 실패: " + (" | ".join(last_errors[-4:]) or "GPU 및 오류 로그를 확인하세요."))

full_text = txt_path.read_text(encoding="utf-8")
if not full_text.strip():
    txt_path.unlink(missing_ok=True)
    show_progress("❌ 비어 있는 인식 결과입니다. TXT 다운로드를 중단했습니다.")
    raise RuntimeError("변환 결과가 비어 있습니다. 오디오 검사 로그를 확인하고 다시 시도해 주세요.")
show_progress(f"✅ 변환 완료! 언어: {detected_language}, {len(full_text):,}자", 100)
preview = html.escape(full_text[:20000])
display(HTML(
    "<div style='margin:18px 0 8px;font-weight:700'>변환 결과 미리보기</div>"
    "<textarea readonly style='width:100%;height:220px;padding:12px;border:1px solid #cbd5e1;"
    "border-radius:8px;background:transparent;color:inherit;resize:vertical'>"
    + preview + "</textarea>"
))
if len(full_text) > 20000:
    print("미리보기는 20,000자까지만 표시합니다. TXT에는 전체 내용이 들어 있습니다.")
print("📄 TXT 파일:", txt_path)
if 자동_TXT_다운로드:
    files.download(str(txt_path))
else:
    print("나중에 STEP 3 셀로 다운로드할 수 있습니다.")





### STEP 3 · TXT 다시 다운로드 (선택)
브라우저에서 자동 다운로드가 차단되었거나, 결과 파일을 다시 다운로드하고 싶을 때만 아래 셀을 실행하세요. 왼쪽 **파일** 탭의 \`/content/whispercolab/results/\` 에서도 TXT를 찾을 수 있습니다.

업로드 창이 작동하지 않는 경우에는 **Colab 왼쪽 파일 탭 → 파일 업로드**로 올린 다음 STEP 2의 \`파일 가져오기\`를 **Colab 파일 경로 사용**으로 바꿔 파일 경로를 입력하세요.



In [ ]:
# @title STEP 3 · 최근 TXT 다시 다운로드 (필요할 때만 ▶)
from pathlib import Path
from google.colab import files

results = sorted(
    Path("/content/whispercolab/results").glob("*.txt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not results:
    print("TXT 결과가 없습니다. 먼저 STEP 2로 음성을 변환하세요.")
else:
    print("다운로드:", results[0].name)
    files.download(str(results[0]))

